# LeetCode 3541 — Find Most Frequent Vowel and Consonant bằng Python

Link đề: https://leetcode.com/problems/find-most-frequent-vowel-and-consonant/ · Độ khó: **Easy** · Chủ đề: Hash Table, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def maxFreqSum(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm tần suất cao nhất trong **các nguyên âm** và tần suất cao nhất trong **các phụ âm**, rồi **cộng hai số đó**; nếu một loại không xuất hiện thì tần suất của nó là `0`.
2. Biết đếm ký tự bằng `collections.Counter`, bằng `dict`, và bằng **mảng 26 ô** với `ord(ch) - ord('a')`.
3. Biết phân loại ký tự bằng tập `"aeiou"` và xử lý `max` trên dãy rỗng bằng `default=0`.
4. Viết 4 cách: `Counter` rồi duyệt, mảng đếm 26 ô, **một lượt duyệt** vừa đếm vừa cập nhật hai giá trị lớn nhất, và `max(..., default=0)` với `str.count`.
5. Tránh các lỗi: cộng **tổng** các tần suất thay vì tần suất **lớn nhất**, lấy tần suất lớn nhất chung không tách nguyên âm/phụ âm, `max` trên dãy rỗng gây `ValueError`, và chỉ lấy ký tự đầu tiên gặp.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm các chữ cái tiếng Anh **thường**. Trong `s`:

- **Nguyên âm** là `a`, `e`, `i`, `o`, `u`; mọi chữ cái còn lại là **phụ âm**.
- Tìm nguyên âm có tần suất (số lần xuất hiện) **lớn nhất**, và phụ âm có tần suất **lớn nhất**.
- Trả về **tổng** của hai tần suất lớn nhất đó. Nếu `s` không có nguyên âm hoặc không có phụ âm thì tần suất lớn nhất của loại đó tính là `0`.

Nếu có nhiều chữ cùng tần suất lớn nhất thì chọn chữ nào cũng được (tổng không đổi).

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"successes"` | `6` | nguyên âm: `u` (1 lần), `e` (2 lần) -> lớn nhất `2`; phụ âm: `s` (4 lần), `c` (2 lần) -> lớn nhất `4`; tổng `2 + 4 = 6` |
| 2 | `"aeiaeia"` | `3` | nguyên âm `a` xuất hiện `3` lần; không có phụ âm (`0`); tổng `3 + 0 = 3` |

Constraints: `1 <= s.length <= 100`; `s` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
from collections import Counter
for s, expected in [("successes", 6), ("aeiaeia", 3)]:
    cnt = Counter(s)
    vowels = {c: n for c, n in cnt.items() if c in "aeiou"}
    cons = {c: n for c, n in cnt.items() if c not in "aeiou"}
    best_v = max(vowels.values(), default=0)
    best_c = max(cons.values(), default=0)
    print(f"s={s!r}: nguyên âm {vowels} -> {best_v}; phụ âm {cons} -> {best_c}; tổng {best_v + best_c} (mong đợi {expected})")

s='successes': nguyên âm {'u': 1, 'e': 2} -> 2; phụ âm {'s': 4, 'c': 2} -> 4; tổng 6 (mong đợi 6)
s='aeiaeia': nguyên âm {'a': 3, 'e': 2, 'i': 2} -> 3; phụ âm {} -> 0; tổng 3 (mong đợi 3)


## 3. Kiến thức nền

### 3.1. Nguyên âm và phụ âm

Trong bài này **nguyên âm** chỉ gồm 5 chữ `a e i o u`. Mọi chữ thường còn lại (kể cả `y`) là **phụ âm**. Kiểm tra thuộc tập bằng `ch in "aeiou"` (với chuỗi) hoặc `ch in {"a","e","i","o","u"}` (với `set`, tra cứu nhanh hơn một chút).

In [2]:
VOWELS = set("aeiou")
for ch in "abeyuz":
    print(ch, "nguyên âm" if ch in VOWELS else "phụ âm")

a nguyên âm
b phụ âm
e nguyên âm
y phụ âm
u nguyên âm
z phụ âm


### 3.2. `collections.Counter` — đếm tần suất

`Counter("successes")` cho `Counter({'s': 4, 'c': 2, 'e': 2, 'u': 1})`. `Counter` là một dạng `dict`: `cnt[ch]` là số lần `ch` xuất hiện (`0` nếu chưa từng thấy, không gây `KeyError`). `.items()` duyệt các cặp `(chữ, số_lần)`, `.values()` chỉ lấy số lần.

In [3]:
from collections import Counter
cnt = Counter("successes")
print(cnt)
print(cnt["s"], cnt["z"])            # 'z' không có -> 0
print(list(cnt.values()))

Counter({'s': 4, 'c': 2, 'e': 2, 'u': 1})
4 0
[4, 1, 2, 2]


### 3.3. Mảng đếm 26 ô và `ord`

Vì chỉ có 26 chữ thường, ta có thể dùng mảng `cnt = [0] * 26` với `cnt[ord(ch) - ord('a')] += 1`. `ord('a') = 97`, nên chữ `'a'` ứng với ô `0`, `'z'` ứng với ô `25`. Mảng đếm nhanh và không cần thư viện.

In [4]:
cnt = [0] * 26
for ch in "successes":
    cnt[ord(ch) - ord('a')] += 1
print({chr(ord('a') + i): c for i, c in enumerate(cnt) if c})

{'c': 2, 'e': 2, 's': 4, 'u': 1}


### 3.4. `max` với `default` và lỗi `ValueError`

`max([])` gây `ValueError: max() arg is an empty sequence`. Dùng `max(iterable, default=0)` để có giá trị mặc định khi dãy rỗng — đúng yêu cầu "không có nguyên âm/phụ âm thì tính `0`".

In [5]:
print(max([3, 1, 2], default=0), max([], default=0))
try:
    max([])
except ValueError as e:
    print("ValueError:", e)

3 0
ValueError: max() iterable argument is empty


### 3.5. `str.count`

`s.count(ch)` đếm số lần ký tự `ch` xuất hiện trong `s` (mỗi lần gọi duyệt cả chuỗi, `O(n)`). Với 26 chữ cái cố định ta gọi tối đa 26 lần: `O(26 · n)`, vẫn là `O(n)`.

In [6]:
s = "successes"
print(s.count("s"), s.count("e"), s.count("z"))
print(max((s.count(c) for c in "aeiou"), default=0))

4 2 0
2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm tay với `"successes"`: đếm từng chữ.

```
s: 4   c: 2   e: 2   u: 1
nguyên âm (e, u):  số lần 2 và 1 -> lớn nhất 2
phụ âm    (s, c):  số lần 4 và 2 -> lớn nhất 4
tổng: 2 + 4 = 6
```

Bài toán gồm hai việc độc lập: (1) **đếm tần suất** mỗi chữ; (2) **tách** thành hai nhóm rồi lấy **tần suất lớn nhất** mỗi nhóm. Với `"aeiaeia"` nhóm phụ âm rỗng nên tần suất lớn nhất của nó là `0` (không phải lỗi).

Điểm hay sai: tổng cần tính là **`max(nguyên âm)` cộng `max(phụ âm)`**, không phải tổng của mọi tần suất (cái đó luôn bằng độ dài chuỗi) và cũng không phải tần suất lớn nhất của cả chuỗi.

## 5. Cách 1 — Brute Force / baseline

Đếm bằng `Counter`, rồi duyệt từng cặp `(chữ, số_lần)`: nếu chữ là nguyên âm thì cập nhật `best_vowel`, ngược lại cập nhật `best_cons`. Cả hai khởi tạo `0` nên trường hợp thiếu một loại tự đúng.

In [7]:
from collections import Counter

class SolutionCounter:
    def maxFreqSum(self, s):
        cnt = Counter(s)
        best_vowel = 0
        best_cons = 0
        for ch, c in cnt.items():
            if ch in "aeiou":
                best_vowel = max(best_vowel, c)
            else:
                best_cons = max(best_cons, c)
        return best_vowel + best_cons

print(SolutionCounter().maxFreqSum("successes"))
print(SolutionCounter().maxFreqSum("aeiaeia"))

6
3


**Độ phức tạp:** gọi `n = len(s)`. `Counter` mất `O(n)`; duyệt tối đa `26` chữ khác nhau mất `O(26) = O(1)`. Tổng `O(n)` thời gian, `O(1)` bộ nhớ (tối đa 26 khóa). Đây đã là tối ưu về bậc.

## 6. Ý tưởng thuật toán / bước đột phá

Bài này không có "nút thắt" cần phá; ta cải thiện **cách viết**:

- **Bỏ thư viện:** mảng `[0] * 26` thay `Counter`, truy cập bằng `ord(ch) - ord('a')`.
- **Một lượt duyệt:** ngay khi tăng `cnt[ch]`, so với `best` của nhóm tương ứng và cập nhật. Không cần duyệt lần hai.
- **Viết ngắn:** `max(..., default=0)` cho từng nhóm.

**Quy tắc "nếu… thì…":**

- Nếu ký tự thuộc `aeiou` thì tần suất của nó chỉ có thể làm tăng `best_vowel`; ngược lại chỉ làm tăng `best_cons`.
- `best_x` luôn bằng `max` của các tần suất hiện tại trong nhóm `x` (invariant), vì mỗi lần một tần suất tăng thêm `1` ta so lại ngay với `best_x`.
- Nhóm không xuất hiện thì `best_x = 0` (giá trị khởi tạo).

In [8]:
s = "successes"
cnt = [0] * 26
best_v = best_c = 0
for ch in s:
    i = ord(ch) - ord('a')
    cnt[i] += 1
    if ch in "aeiou":
        best_v = max(best_v, cnt[i])
    else:
        best_c = max(best_c, cnt[i])
    print(f"đọc {ch!r}: cnt={cnt[i]} best_vowel={best_v} best_cons={best_c}")
print(best_v + best_c)

đọc 's': cnt=1 best_vowel=0 best_cons=1
đọc 'u': cnt=1 best_vowel=1 best_cons=1
đọc 'c': cnt=1 best_vowel=1 best_cons=1
đọc 'c': cnt=2 best_vowel=1 best_cons=2
đọc 'e': cnt=1 best_vowel=1 best_cons=2
đọc 's': cnt=2 best_vowel=1 best_cons=2
đọc 's': cnt=3 best_vowel=1 best_cons=3
đọc 'e': cnt=2 best_vowel=2 best_cons=3
đọc 's': cnt=4 best_vowel=2 best_cons=4
6


## 7. Các cách tiếp theo

Ta có 4 cách: (1) `Counter` + duyệt; (2) mảng đếm 26 ô; (3) một lượt duyệt cập nhật hai giá trị lớn nhất; (4) `max(..., default=0)` với `str.count`.

### Cách 2 — Mảng đếm 26 ô

Đếm vào `cnt[0..25]`; sau đó duyệt chỉ số `i = 0..25`, đổi sang chữ `chr(ord('a') + i)` để xét nguyên âm/phụ âm.

In [9]:
class SolutionArray:
    def maxFreqSum(self, s):
        cnt = [0] * 26
        for ch in s:
            cnt[ord(ch) - ord('a')] += 1
        best_vowel = best_cons = 0
        for i in range(26):
            letter = chr(ord('a') + i)
            if letter in "aeiou":
                best_vowel = max(best_vowel, cnt[i])
            else:
                best_cons = max(best_cons, cnt[i])
        return best_vowel + best_cons

print(SolutionArray().maxFreqSum("successes"))
print(SolutionArray().maxFreqSum("aeiaeia"))

6
3


Độ phức tạp Cách 2: `O(n + 26) = O(n)` thời gian, `O(26) = O(1)` bộ nhớ.

### Cách 3 — Một lượt duyệt

Vừa đếm vừa giữ `best_vowel`, `best_cons`. Mỗi ký tự: tăng đếm, rồi cập nhật `best` của nhóm ký tự đó. Không cần duyệt lần hai.

In [10]:
class SolutionOnePass:
    def maxFreqSum(self, s):
        cnt = {}
        best_vowel = best_cons = 0
        for ch in s:
            cnt[ch] = cnt.get(ch, 0) + 1
            if ch in "aeiou":
                if cnt[ch] > best_vowel:
                    best_vowel = cnt[ch]
            else:
                if cnt[ch] > best_cons:
                    best_cons = cnt[ch]
        return best_vowel + best_cons

print(SolutionOnePass().maxFreqSum("successes"))
print(SolutionOnePass().maxFreqSum("aeiaeia"))

6
3


Độ phức tạp Cách 3: `O(n)` thời gian, `O(1)` bộ nhớ (tối đa 26 khóa trong `dict`).

### Cách 4 — `max(..., default=0)` với `str.count`

Với từng nhóm, lấy `max` của `s.count(c)` qua các chữ của nhóm. Nhóm nguyên âm duyệt `"aeiou"`, nhóm phụ âm duyệt 21 chữ còn lại. `default=0` không thực sự cần ở đây (vì `count` cho `0` chứ không rỗng) nhưng là thói quen tốt; viết gọn nhất trong bốn cách.

In [11]:
import string

class SolutionCount:
    def maxFreqSum(self, s):
        vowels = "aeiou"
        consonants = [c for c in string.ascii_lowercase if c not in vowels]
        return (max((s.count(c) for c in vowels), default=0)
                + max((s.count(c) for c in consonants), default=0))

print(SolutionCount().maxFreqSum("successes"))
print(SolutionCount().maxFreqSum("aeiaeia"))

6
3


Độ phức tạp Cách 4: `26` lần `count`, mỗi lần `O(n)` -> `O(26 · n) = O(n)` thời gian, `O(1)` bộ nhớ.

## 8. TRACE từng bước

Trace Cách 3 (một lượt) với `s = "successes"`:

| bước | ký tự | `cnt[ch]` | nhóm | `best_vowel` | `best_cons` |
|---|---|---|---|---|---|
| 1 | `s` | 1 | phụ âm | 0 | 1 |
| 2 | `u` | 1 | nguyên âm | 1 | 1 |
| 3 | `c` | 1 | phụ âm | 1 | 1 |
| 4 | `c` | 2 | phụ âm | 1 | 2 |
| 5 | `e` | 1 | nguyên âm | 1 | 2 |
| 6 | `s` | 2 | phụ âm | 1 | 2 |
| 7 | `s` | 3 | phụ âm | 1 | 3 |
| 8 | `e` | 2 | nguyên âm | 2 | 3 |
| 9 | `s` | 4 | phụ âm | 2 | 4 |

Kết quả `2 + 4 = 6`.

In [12]:
def trace_one_pass(s):
    cnt = {}
    best_vowel = best_cons = 0
    for step, ch in enumerate(s, start=1):
        cnt[ch] = cnt.get(ch, 0) + 1
        group = "nguyên âm" if ch in "aeiou" else "phụ âm"
        if ch in "aeiou":
            best_vowel = max(best_vowel, cnt[ch])
        else:
            best_cons = max(best_cons, cnt[ch])
        print(f"bước {step}: {ch!r} cnt={cnt[ch]} ({group}) best_vowel={best_vowel} best_cons={best_cons}")
    print("tổng =", best_vowel + best_cons)
    return best_vowel + best_cons

assert trace_one_pass("successes") == 6
assert trace_one_pass("aeiaeia") == 3

bước 1: 's' cnt=1 (phụ âm) best_vowel=0 best_cons=1
bước 2: 'u' cnt=1 (nguyên âm) best_vowel=1 best_cons=1
bước 3: 'c' cnt=1 (phụ âm) best_vowel=1 best_cons=1
bước 4: 'c' cnt=2 (phụ âm) best_vowel=1 best_cons=2
bước 5: 'e' cnt=1 (nguyên âm) best_vowel=1 best_cons=2
bước 6: 's' cnt=2 (phụ âm) best_vowel=1 best_cons=2
bước 7: 's' cnt=3 (phụ âm) best_vowel=1 best_cons=3
bước 8: 'e' cnt=2 (nguyên âm) best_vowel=2 best_cons=3
bước 9: 's' cnt=4 (phụ âm) best_vowel=2 best_cons=4
tổng = 6
bước 1: 'a' cnt=1 (nguyên âm) best_vowel=1 best_cons=0
bước 2: 'e' cnt=1 (nguyên âm) best_vowel=1 best_cons=0
bước 3: 'i' cnt=1 (nguyên âm) best_vowel=1 best_cons=0
bước 4: 'a' cnt=2 (nguyên âm) best_vowel=2 best_cons=0
bước 5: 'e' cnt=2 (nguyên âm) best_vowel=2 best_cons=0
bước 6: 'i' cnt=2 (nguyên âm) best_vowel=2 best_cons=0
bước 7: 'a' cnt=3 (nguyên âm) best_vowel=3 best_cons=0
tổng = 3


## 9. Code nộp LeetCode

In [13]:
from collections import Counter

class Solution:
    def maxFreqSum(self, s: str) -> int:
        cnt = Counter(s)
        best_vowel = 0
        best_cons = 0
        for ch, c in cnt.items():
            if ch in "aeiou":
                best_vowel = max(best_vowel, c)
            else:
                best_cons = max(best_cons, c)
        return best_vowel + best_cons

print(Solution().maxFreqSum("successes"), Solution().maxFreqSum("aeiaeia"))

6 3


## 10. Vì sao thuật toán đúng

- **Đếm đúng:** `Counter`/mảng/`dict`/`count` đều cho `cnt[ch]` = số lần `ch` xuất hiện (mỗi ký tự của `s` được đếm đúng một lần).
- **Phân nhóm đúng:** mỗi chữ cái thường thuộc đúng một nhóm (nguyên âm nếu thuộc `aeiou`, còn lại là phụ âm), không bỏ sót, không trùng.
- **Lấy max đúng:** `best_vowel` = `max` các `cnt[ch]` với `ch` nguyên âm; nếu không có nguyên âm nào xuất hiện thì giá trị khởi tạo `0` đúng như đề quy định. Tương tự `best_cons`. Đề chỉ cần **giá trị** tần suất lớn nhất, nên khi có nhiều chữ hòa không ảnh hưởng.
- **Cách 3:** sau mỗi ký tự, `best_x` bằng max hiện tại của nhóm. Chỉ tần suất của ký tự vừa đọc thay đổi, nên chỉ cần so riêng nó với `best` của nhóm (invariant).
- **Tách biệt:** tổng cần tính là `max` + `max`, không phải tổng hay max chung; các test ở mục 12 và các lỗi ở mục 13 bảo vệ điều này.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. `Counter` + duyệt | `O(n)` | `O(1)` (<= 26 khóa) |
| 2. Mảng 26 ô | `O(n + 26)` | `O(1)` |
| 3. Một lượt duyệt | `O(n)` | `O(1)` |
| 4. `max` + `str.count` | `O(26 · n)` | `O(1)` |

Với `n <= 100` mọi cách tức thì. Khi `n` rất lớn, Cách 1–3 tốt hơn Cách 4 một hằng số `~26`.

## 12. Bộ test

Test 1: ví dụ đề và các edge case (một chữ, toàn nguyên âm, toàn phụ âm, `y`, hòa tần suất, độ dài `100`). Mọi input tuân thủ `1 <= n <= 100`, chữ thường.

In [14]:
impls = [SolutionCounter(), SolutionArray(), SolutionOnePass(), SolutionCount(), Solution()]
tests = [
    ("successes", 6),
    ("aeiaeia", 3),
    ("a", 1),                       # n = 1, chỉ nguyên âm
    ("b", 1),                       # n = 1, chỉ phụ âm
    ("y", 1),                       # y là phụ âm
    ("aeiou", 1),                   # toàn nguyên âm, mỗi chữ 1 lần
    ("bcdfg", 1),                   # toàn phụ âm, mỗi chữ 1 lần
    ("aabb", 4),                    # nguyên âm a:2 + phụ âm b:2
    ("abab", 4),                    # a:2 + b:2, đan xen
    ("aeaeaebbcc", 5),              # a:3, e:3 -> 3; b:2, c:2 -> 2; tổng 5
    ("zzzzzy", 5),                  # y và z đều phụ âm: max 5; không có nguyên âm
    ("a" * 100, 100),               # biên trên n, toàn một chữ nguyên âm
    ("b" * 99 + "a", 100),          # 99 phụ âm b + 1 nguyên âm a
]
for s, exp in tests:
    assert 1 <= len(s) <= 100 and s.islower() and s.isalpha()
    for im in impls:
        got = im.maxFreqSum(s)
        assert got == exp, (type(im).__name__, s, got, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 13


Test 2: ngẫu nhiên có seed. Oracle độc lập: sắp xếp các chữ, tách nhóm bằng hai chuỗi con `"aeiou"` và phần bù, rồi lấy `max` bằng vòng lặp (không dùng `Counter`, mảng 26 ô hay `max(default=...)`).

In [15]:
import random
import string
random.seed(3541)

def oracle(s):
    best_v = 0
    best_c = 0
    for letter in set(s):
        times = 0
        for ch in s:
            if ch == letter:
                times += 1
        if letter in ("a", "e", "i", "o", "u"):
            if times > best_v:
                best_v = times
        else:
            if times > best_c:
                best_c = times
    return best_v + best_c

for t in range(3000):
    n = random.randint(1, 100)
    pool = random.choice(["ab", "aeiou", "bcdfg", string.ascii_lowercase, "aeiouzy"])
    s = "".join(random.choice(pool) for _ in range(n))
    assert 1 <= len(s) <= 100 and s.islower()
    ref = oracle(s)
    for im in impls:
        assert im.maxFreqSum(s) == ref, (type(im).__name__, s)
print("3000 test ngẫu nhiên OK")

3000 test ngẫu nhiên OK


Test 3: tính chất — hoán vị thứ tự ký tự không đổi kết quả; kết quả `<= n`; kết quả `>= 1`; và kết quả bằng `n` khi chuỗi chỉ có **một** chữ cái duy nhất hoặc khi (một nguyên âm duy nhất và một phụ âm duy nhất) phủ hết chuỗi.

In [16]:
random.seed(41)
for _ in range(1000):
    n = random.randint(1, 100)
    s = "".join(random.choice(string.ascii_lowercase) for _ in range(n))
    base = Solution().maxFreqSum(s)
    chars = list(s)
    random.shuffle(chars)
    assert Solution().maxFreqSum("".join(chars)) == base
    assert 1 <= base <= n
for _ in range(200):
    n = random.randint(1, 100)
    k = random.randint(0, n)
    v = random.choice("aeiou")
    c = random.choice([x for x in string.ascii_lowercase if x not in "aeiou"])
    s = v * k + c * (n - k)                      # chỉ một nguyên âm và một phụ âm
    assert Solution().maxFreqSum(s) == n
print("tính chất (hoán vị, 1 <= kết quả <= n, hai chữ phủ hết chuỗi): OK")

tính chất (hoán vị, 1 <= kết quả <= n, hai chữ phủ hết chuỗi): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: cộng **tổng** các tần suất thay vì tần suất **lớn nhất**

In [17]:
def wrong_sum_all(s):
    cnt = Counter(s)
    vowel_total = sum(c for ch, c in cnt.items() if ch in "aeiou")      # SAI: tổng, không phải max
    cons_total = sum(c for ch, c in cnt.items() if ch not in "aeiou")
    return vowel_total + cons_total

s = "successes"
print("sai :", wrong_sum_all(s))
print("đúng:", Solution().maxFreqSum(s))
assert wrong_sum_all(s) != Solution().maxFreqSum(s)

sai : 9
đúng: 6


Vì sao sai: tổng mọi tần suất luôn bằng `len(s) = 9`, trong khi đề hỏi **tần suất lớn nhất** của mỗi nhóm (`2 + 4 = 6`). **Sửa:** dùng `max` trong mỗi nhóm.

### Lỗi 2: lấy tần suất lớn nhất chung, không tách nguyên âm/phụ âm

In [18]:
def wrong_global_max(s):
    return max(Counter(s).values())                # SAI: không tách hai nhóm, không cộng

s = "successes"
print("sai :", wrong_global_max(s))
print("đúng:", Solution().maxFreqSum(s))
assert wrong_global_max(s) != Solution().maxFreqSum(s)

sai : 4
đúng: 6


Vì sao sai: `max` chung là `4` (chữ `s`), thiếu phần của nguyên âm. Đề cần **cả hai** (một nguyên âm, một phụ âm). **Sửa:** tính `best_vowel` và `best_cons` riêng rồi cộng.

### Lỗi 3: `max` trên dãy rỗng gây `ValueError`

In [19]:
def wrong_empty_max(s):
    cnt = Counter(s)
    best_vowel = max(c for ch, c in cnt.items() if ch in "aeiou")          # SAI: có thể rỗng
    best_cons = max(c for ch, c in cnt.items() if ch not in "aeiou")       # SAI: có thể rỗng
    return best_vowel + best_cons

s = "aeiaeia"                                         # không có phụ âm
try:
    wrong_empty_max(s)
    raised = False
except ValueError:
    raised = True
print("code sai có bị ValueError với", repr(s), ":", raised)
print("đúng:", Solution().maxFreqSum(s))
assert raised and Solution().maxFreqSum(s) == 3

code sai có bị ValueError với 'aeiaeia' : True
đúng: 3


Vì sao sai: không có phụ âm nên generator rỗng, `max` ném `ValueError`. Đề quy định nhóm vắng mặt tính `0`. **Sửa:** `max(..., default=0)` hoặc khởi tạo biến `best = 0` rồi cập nhật.

### Lỗi 4: chỉ lấy tần suất của chữ đầu tiên gặp trong mỗi nhóm

In [20]:
def wrong_first_seen(s):
    cnt = Counter(s)
    first_v = next((c for ch, c in cnt.items() if ch in "aeiou"), 0)       # SAI: chữ đầu tiên, chưa chắc nhiều nhất
    first_c = next((c for ch, c in cnt.items() if ch not in "aeiou"), 0)
    return first_v + first_c

s = "successes"
print("sai :", wrong_first_seen(s))
print("đúng:", Solution().maxFreqSum(s))
assert wrong_first_seen(s) != Solution().maxFreqSum(s)

sai : 5
đúng: 6


Vì sao sai: `Counter` giữ thứ tự xuất hiện: nguyên âm đầu tiên là `u` (1 lần), không phải `e` (2 lần, lớn nhất), nên ra `1 + 4 = 5`. **Sửa:** duyệt hết và lấy `max`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tần suất lớn nhất" + chuỗi chỉ gồm 26 chữ thường -> `Counter` hoặc mảng 26 ô.
- Hai nhóm (nguyên âm / phụ âm) -> hai biến `best_vowel`, `best_cons` khởi tạo `0`.
- Nhóm có thể **vắng mặt**: khởi tạo `0` (hoặc `default=0`), tránh `max` của dãy rỗng.
- Nhớ nguyên âm chỉ `aeiou`; `y` là phụ âm trong bài này.
- Hòa tần suất không ảnh hưởng đáp án (chỉ cần **giá trị** lớn nhất).
- Bài họ hàng: `387. First Unique Character in a String`, `1189. Maximum Number of Balloons`; `345. Reverse Vowels of a String`.

## 15. Biến thể của bài

1. **Chữ hoa lẫn chữ thường:** chuẩn hóa bằng `s.lower()` trước khi đếm.
2. Trả về **chính hai chữ** (nguyên âm và phụ âm) thay vì tổng; nếu hòa chọn chữ nhỏ nhất theo thứ tự bảng chữ cái.
3. **Tần suất lớn thứ hai** của mỗi nhóm.
4. Cho phép bộ nguyên âm **tùy biến** (ví dụ coi `y` là nguyên âm) — đổi một hằng số.
5. Dữ liệu là **luồng** rất dài: Cách 3 (một lượt, bộ nhớ `O(1)`) là cách duy nhất phù hợp.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `Counter` + duyệt | đếm rồi tách hai nhóm | `O(n)` | `O(1)` | **rõ ràng, nên nộp** |
| 2. Mảng 26 ô | `ord(ch) - ord('a')` | `O(n + 26)` | `O(1)` | không dùng thư viện |
| 3. Một lượt duyệt | vừa đếm vừa cập nhật `best` | `O(n)` | `O(1)` | luồng dữ liệu, tối ưu bộ nhớ |
| 4. `max` + `str.count` | `count` cho từng chữ | `O(26n)` | `O(1)` | viết ngắn nhất |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `maxFreqSum("leetcode")` bằng bao nhiêu?

**Bài 2 (tính tay):** `s = "programming"`: tần suất lớn nhất của nguyên âm, của phụ âm, và tổng.

**Bài 3 (điền TODO):** hoàn thành `is_vowel(ch)` trả về `True` nếu `ch` là nguyên âm `aeiou`.

**Bài 4 (tự viết lại):** viết `most_frequent_letters(s)` trả về cặp `(nguyên_âm, phụ_âm)` có tần suất lớn nhất (hòa thì chọn chữ nhỏ hơn theo thứ tự bảng chữ cái; `None` nếu nhóm vắng).

**Bài 5 (mini challenge):** viết `max_freq_sum_y_vowel(s)` giống bài gốc nhưng coi **`y` cũng là nguyên âm**.

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().maxFreqSum("leetcode"))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def is_vowel(ch):
    # TODO: trả về True nếu ch thuộc "aeiou"
    return False

4


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** đếm từng chữ; tách hai nhóm; lấy lớn nhất mỗi nhóm rồi cộng.

**Mức 2 (vừa):** Bài 3: `return ch in "aeiou"`. Bài 4: duyệt các chữ theo thứ tự `sorted(cnt)` và chỉ cập nhật khi tần suất **lớn hơn nghiêm ngặt**, để hòa thì giữ chữ đứng trước (nhỏ hơn). Bài 5: đổi tập nguyên âm thành `"aeiouy"`.

**Mức 3 (gần đáp án):** Bài 1: `leetcode` có `e` xuất hiện 3 lần (nguyên âm: `e`:3, `o`:1) -> `3`; phụ âm: `l`:1, `t`:1, `c`:1, `d`:1 -> `1`; tổng `4`. Bài 2: `programming`: nguyên âm `o`:1, `a`:1, `i`:1 -> `1`; phụ âm `r`:2, `g`:2, `m`:2, `p`:1, `n`:1 -> `2`; tổng `3`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
assert Solution().maxFreqSum("leetcode") == 4

# Bài 2
assert Solution().maxFreqSum("programming") == 3

# Bài 3
def is_vowel(ch):
    return ch in "aeiou"
assert is_vowel("a") and is_vowel("u") and not is_vowel("y") and not is_vowel("b")

# Bài 4
def most_frequent_letters(s):
    cnt = Counter(s)
    best_v = best_c = None
    for ch in sorted(cnt):                            # chữ nhỏ trước; chỉ thay khi LỚN HƠN nghiêm ngặt
        if is_vowel(ch):
            if best_v is None or cnt[ch] > cnt[best_v]:
                best_v = ch
        else:
            if best_c is None or cnt[ch] > cnt[best_c]:
                best_c = ch
    return best_v, best_c
assert most_frequent_letters("successes") == ("e", "s")
assert most_frequent_letters("aeiaeia") == ("a", None)
assert most_frequent_letters("bcd") == (None, "b")        # hòa: chọn chữ nhỏ nhất

# Bài 5
def max_freq_sum_y_vowel(s):
    cnt = Counter(s)
    bv = max((c for ch, c in cnt.items() if ch in "aeiouy"), default=0)
    bc = max((c for ch, c in cnt.items() if ch not in "aeiouy"), default=0)
    return bv + bc
assert max_freq_sum_y_vowel("yyz") == 2 + 1               # y nguyên âm: 2, z phụ âm: 1
assert Solution().maxFreqSum("yyz") == 2                  # bài gốc: y, z đều phụ âm -> max 2, không có nguyên âm
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "max tần suất theo hai nhóm":**

```python
cnt = Counter(s)
best_vowel = best_cons = 0
for ch, c in cnt.items():
    if ch in "aeiou":
        best_vowel = max(best_vowel, c)
    else:
        best_cons = max(best_cons, c)
return best_vowel + best_cons
```

**Checklist trước khi submit:**

- [ ] Nguyên âm chỉ `aeiou` (`y` là phụ âm)?
- [ ] Lấy **max** trong mỗi nhóm (không phải tổng, không phải max chung)?
- [ ] Nhóm vắng mặt cho `0` (không để `max` của dãy rỗng gây lỗi)?
- [ ] Đã thử chuỗi 1 chữ, toàn nguyên âm (`"aeiaeia"`), toàn phụ âm, độ dài `100`?
- [ ] Chữ ký đúng `maxFreqSum(self, s: str) -> int`?